# XRF spectrum with monochromatic excitation: identification and fit

Analysis of `XRFSpectrum.mca`, the PyMca training spectrum
([PyMca XRF analysis tutorial](https://www.silx.org/doc/PyMca/5.5.0/training/quantification/index.html)):
a multi-element thin standard, simplified as **100 µm of water containing 500 ppm Co**,
excited by a **monochromatised Mo tube** (Mo Kα1/Kα2, ~17.4 keV).

The spectrum itself is not part of this repository: it comes with the PyMca training
material (see the tutorial above). Put `XRFSpectrum.mca` next to this notebook to run it.

* Detector: Si, 450 µm, 8 µm Be window, 2 mm of air to the sample.
* Geometry: incident beam at 0.1° from the surface, detector at 90° (take-off).
* Calibration: the main Co line is close to channel 1474; the detector has a large offset,
  so a second point is needed — the elastic peak at the excitation energy.

**Aim:** identify every element present and fit the whole spectrum, scatter peaks included.

The model (`xrf_fit`) places every line of every element at its tabulated energy with
relative intensities computed for this excitation, detector and sample, and adds:

* a detector response with low-energy tail and step (Hypermet-type), shared by all lines;
* Si escape peaks;
* the **elastic** doublet (Mo Kα1/Kα2) and the **Compton** peak in the impulse
  approximation — the Compton profile J(p<sub>z</sub>) of water from xraylib, averaged over the
  range of scattering angles the detector collects (a detector 2 mm from the sample sees a
  wide range, which makes the Compton peak broad and asymmetric);
* a polynomial background.

PyMca's tutorial stops the fit at 16.3 keV because its scatter peaks are approximate; here the
fit runs up to the elastic peak.

In [ ]:
%matplotlib inline
import json
import logging
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xraylib
from scipy.signal import find_peaks

from xrf_beamline import monochromatic_spectrum, tube_lines
from xrf_calibration import calibration_from_peaks
from xrf_fit import (Calibration, Detector, ScatterSource, element_line_groups,
                     fit_average_spectrum, identify_peaks, plot_fit)
from xrf_matrix import Geometry, Layer, MatrixModel

logging.basicConfig(level=logging.WARNING)

## 1. Settings

In [ ]:
MCA_FILE = Path("XRFSpectrum.mca")
OUT_DIR = Path("processed")
OUT_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------------- excitation
ANODE = "Mo"                       # monochromatised tube: its K-alpha1/K-alpha2 doublet
E_EXC, W_EXC = tube_lines(ANODE)   # eV, relative intensities

# ---------------------------------------------------------------- detector
# Si 450 um, 8 um Be window, 2 mm air. The resolution is only a starting value (fitted).
DETECTOR0 = Detector.from_fwhm(fwhm_ev=150.0, at_energy_ev=5900.0, material="Si", fano=0.12,
                               gamma_ev=0.0, sensor_thickness_um=450.0,
                               window_material="Be", window_density_g_cm3=1.848,
                               window_thickness_um=8.0, air_path_cm=0.2)

# ---------------------------------------------------------------- sample and geometry
CO_PPM = 500.0                      # mass fraction of Co in water, ppm (internal standard)
w = CO_PPM * 1e-6
co_per_h2o = (w / xraylib.AtomicWeight(27)) / ((1 - w) / 18.015)
SAMPLE = Layer({"H": 2, "O": 1, "Co": co_per_h2o}, 1.0, 100.0, "Co in water")
GEOMETRY = Geometry(incidence_deg=0.1, takeoff_deg=90.0)
SCATTER_ANGLE = 90.0 + GEOMETRY.incidence_deg   # between the beam and the detector direction
SCATTERER = "H2O"                                # Compton profile of the scattering medium

# ---------------------------------------------------------------- calibration
CO_CHANNEL = 1474                   # main Co line, close to this channel

# ---------------------------------------------------------------- fit
ROI_EV = (1200.0, 17900.0)          # from above the noise/discriminator edge to the elastic peak
BKG_ORDER = 4
FAMILIES = ("K", "L", "M")
MIN_REL = 0.005

## 2. The spectrum

In [ ]:
y = np.loadtxt(MCA_FILE, comments="#")
yerr = np.sqrt(np.maximum(y, 1.0))
ch = np.arange(y.size)
print(f"{y.size} channels, {y.sum():.3e} counts")

fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogy(ch, np.maximum(y, 0.5), "k-", lw=0.6)
ax.set_xlabel("channel"); ax.set_ylabel("counts")
ax.set_title(f"{MCA_FILE.name}: raw spectrum")
fig.tight_layout()

## 3. Energy calibration

Two reference peaks, each located by a local Gaussian fit:

* **Co Kα** near channel 1474 (Kα1/Kα2 unresolved: their intensity-weighted energy);
* the **elastic peak**, the highest-energy sharp peak of the spectrum, at the excitation
  energy (Mo Kα1/Kα2, weighted the same way).

The calibration is refined later by the full fit, using every line.

In [ ]:
e_co, w_co = tube_lines("Co")                # Co K-alpha1/K-alpha2 (same helper as the tube)
co_ka = float(e_co @ w_co)
e_elastic = float(E_EXC @ W_EXC)

# elastic: the right-most prominent peak of the (lightly smoothed) spectrum
smooth = np.convolve(y, np.ones(9) / 9, mode="same")
pk, _ = find_peaks(smooth, prominence=0.05 * smooth[ch > 500].max())
elastic_ch = int(pk[-1])

CAL0, found = calibration_from_peaks(y, [(CO_CHANNEL, co_ka), (elastic_ch, e_elastic)])
for f, name in zip(found, ("Co Ka", f"elastic ({ANODE} Ka)")):
    print(f"{name:18s} channel {f['channel']:8.2f} +/- {f['channel_err']:.2f}  ->  {f['energy_ev']:8.1f} eV")
print(f"\nE = {CAL0.gain:.4f} eV/ch x channel {CAL0.offset:+.1f} eV"
      f"   (PyMca tutorial: A ~ -0.5 keV, B ~ 0.005 keV/channel)")

## 4. Element identification

### 4a. Peak scouting

`identify_peaks` lists the tabulated lines close to each peak. A coincidence in energy is
only a hint: an element is confirmed when its whole line series fits at the computed ratios.

In [ ]:
roi0 = CAL0.channel_range(*ROI_EV, y.size)
print(f"{'E (keV)':>8} {'height':>8}   candidates")
for p in identify_peaks(y, CAL0, roi0, prominence_frac=0.01, tol_ev=60):
    if p["energy_ev"] < 14600:          # the scatter region comes later
        print(f"{p['energy_ev']/1000:8.3f} {p['height']:8.0f}   {', '.join(p['candidates'][:3])}")

In [ ]:
# Line groups for this excitation, detector and sample (water layer, grazing incidence).
E_GRID, PHI = monochromatic_spectrum(E_EXC, W_EXC)
MATRIX = MatrixModel(SAMPLE, GEOMETRY, E_GRID, PHI, secondary=False)   # dilute: no secondary fluorescence

def make_groups(elements, roi_ev=ROI_EV):
    """K lines, L lines above 1.3 keV, and the M lines of the heavy elements."""
    out = []
    for g in element_line_groups(elements, *roi_ev, families=FAMILIES, min_rel=MIN_REL,
                                 detector=DETECTOR0, matrix=MATRIX):
        z = xraylib.SymbolToAtomicNumber(g.element)
        if g.family == "K" or (g.family == "L" and g.reference.energy_ev > 1300) or (g.family == "M" and z >= 72):
            out.append(g)
    return out

### 4b. First pass, below the scatter peaks

A fast fit over 1.2–14.6 keV with the elements suggested by the scouting (the K-line series,
Ar from the air, …). Its residuals show what is missing.

In [ ]:
FIRST = ["S", "Ar", "K", "Ca", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "As", "Se", "Sr"]
roi1 = (ROI_EV[0], 14600.0)
t0 = time.time()
r1 = fit_average_spectrum(y, make_groups(FIRST, roi1), CAL0, DETECTOR0, CAL0.channel_range(*roi1, y.size),
                          yerr=yerr, background_order=BKG_ORDER, escape=True, fit_calibration=True,
                          fit_tails=True, fit_gamma=False)
print(f"first pass: reduced chi2 = {r1.chi2_red:.1f}  ({time.time()-t0:.0f} s)\n")
res = np.zeros_like(y); res[r1.channels.astype(int)] = r1.residual
print("largest positive residuals:")
for p in sorted(identify_peaks(np.clip(res, 0, None), r1.calibration, (int(r1.channels[0]), int(r1.channels[-1])),
                               prominence_frac=0.2, tol_ev=50), key=lambda p: -p["height"])[:6]:
    print(f"  {p['energy_ev']/1000:6.3f} keV  +{p['height']:5.0f}   {', '.join(p['candidates'][:3])}")
fig, _ = plot_fit(r1, logy=True, data_label="data", ylabel="counts", figsize=(12, 6))
fig.axes[0].legend(ncol=4, fontsize=7)

The two large residuals at **10.27 and 12.21 keV** are the Lα1 and Lβ1 lines of **thallium**
(and its Lγ1 at 14.29 keV); the 10.5 and 12.6 keV peaks also need **lead** (Lα 10.55, Lβ 12.61)
beside As Kα. Their M lines then contribute near 2.3 keV, next to S Kα.

## 5. Full fit, scatter peaks included

The final element list (the 18 elements the PyMca tutorial hints at, plus the Mo L lines — see below), over 1.2–17.9 keV. Free:
calibration (gain, offset), resolution, detector tail and step, the elastic and Compton
amplitudes, the Compton angle and angular spread, an extra Compton broadening and tail.

In [ ]:
ELEMENTS = FIRST + ["Tl", "Pb", "Mo"]   # Mo: L lines only (17.4 keV is below its K edge)
GROUPS = make_groups(ELEMENTS)
SCATTER = ScatterSource(E_EXC, W_EXC, angle_deg=SCATTER_ANGLE, scatterer=SCATTERER)
t0 = time.time()
result = fit_average_spectrum(y, GROUPS, CAL0, DETECTOR0, CAL0.channel_range(*ROI_EV, y.size), yerr=yerr,
                              background_order=BKG_ORDER, escape=True, fit_calibration=True,
                              fit_tails=True, fit_gamma=False, scatter=SCATTER, maxfev=40000)
P = result.params()
CAL = result.calibration
model = result.model
print(f"reduced chi2 = {result.chi2_red:.2f}  ({time.time()-t0:.0f} s)")
print(f"background: {model.background(result.popt).min():.0f} to {model.background(result.popt).max():.0f} counts")

In [ ]:
fig, (ax, axr) = plot_fit(result, logy=True, data_label="data", ylabel="counts", figsize=(13, 7.5))
ax.legend(ncol=5, fontsize=7)
ax.set_title(f"{MCA_FILE.name}: {len(ELEMENTS)} elements + elastic and Compton scatter, "
             f"reduced $\\chi^2$ = {result.chi2_red:.2f}")
fig.savefig(OUT_DIR / f"{MCA_FILE.stem}_fit.png", dpi=150, bbox_inches="tight")

In [ ]:
# Close-ups, linear scale: data, model, and the main components
e_keV = CAL.energy(result.channels) / 1000
fluo = sum(model.group_profile(i, result.popt) for i in range(len(model.groups)))
comp = model.scatter_profile(result.popt, "compton")
elas = model.scatter_profile(result.popt, "elastic")
bkg = model.background(result.popt)
regions = [(1.2, 4.2, "light elements and the 2.3 keV region"), (4.2, 9.5, "Ti to Zn"),
           (9.5, 15.0, "As, Se, Tl, Pb, Sr"), (14.5, 17.9, "scatter peaks")]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (lo, hi, title) in zip(axes.ravel(), regions):
    m = (e_keV >= lo) & (e_keV <= hi)
    ax.plot(e_keV[m], result.y[m], "k.", ms=2, label="data")
    ax.plot(e_keV[m], result.yfit[m], "r-", lw=1.2, label="fit")
    ax.plot(e_keV[m], (bkg + fluo)[m], "-", color="C0", lw=0.8, label="background + fluorescence")
    if hi > 14:
        ax.plot(e_keV[m], (bkg + comp)[m], "-", color="C2", lw=0.8, label="background + Compton")
        ax.plot(e_keV[m], (bkg + elas)[m], "-", color="C1", lw=0.8, label="background + elastic")
    ax.plot(e_keV[m], bkg[m], "--", color="0.5", lw=0.8, label="background")
    ax.set_title(title, loc="left", fontsize=10); ax.set_xlabel("Energy (keV)")
    ax.legend(fontsize=7)
fig.tight_layout()

In [ ]:
print("calibration and detector")
print(f"  E = {CAL.gain:.5f} (+/- {P['cal_gain'][1]:.5f}) x channel {CAL.offset:+.2f} (+/- {P['cal_offset_ev'][1]:.2f}) eV")
print(f"  FWHM: {DETECTOR0.fwhm_ev(5900, noise_ev=P['noise_ev'][0], fano=P['fano'][0]):.0f} eV at 5.9 keV, "
      f"{DETECTOR0.fwhm_ev(10000, noise_ev=P['noise_ev'][0], fano=P['fano'][0]):.0f} eV at 10 keV")
print(f"  tail fraction {P['tail_fraction'][0]:.3f}, slope {P['tail_slope'][0]:.2f} sigma; step {P['step_fraction'][0]:.2e}")
print("scatter")
print(f"  elastic area {P['A_elastic'][0]:.3g}, extra width {P['elastic_sigma_ev'][0]:.0f} eV")
print(f"  Compton area {P['A_compton'][0]:.3g}: angle {P['compton_angle_deg'][0]:.1f} +/- "
      f"{P['compton_angle_spread_deg'][0]:.1f} deg (geometry: {SCATTER_ANGLE:.1f}), "
      f"extra width {P['compton_sigma_ev'][0]:.0f} eV, tail {P['compton_tail_fraction'][0]:.2f}")
print(f"  Compton / elastic = {P['A_compton'][0] / P['A_elastic'][0]:.1f}")

The Compton peak needs a wide range of scattering angles: a detector 2 mm from the sample
collects photons scattered over several tens of degrees, and since the Compton energy depends
non-linearly on the angle the peak becomes broad and asymmetric. The water Compton profile
then accounts for the rest of its shape.

In [ ]:
print(f"{'group':7s}{'E ref (keV)':>12}{'area':>10}{'error':>8}{'S/N':>8}")
for g in GROUPS:
    a, e = result.areas[g.label], result.area_errors[g.label]
    print(f"{g.label:7s}{g.reference.energy_ev/1000:12.3f}{a:10.0f}{e:8.0f}{a/max(e, 1e-9):8.1f}")

## 6. Candidates tested and rejected

Elements that could explain small residuals, added to the final model one batch at a time: an
element is kept if its signal is significant (well above 3 σ) and consistent across its lines.

**Mo** was found this way (its L lines at about 4 σ) and is now in the final list. Mo is the
tube anode, and Mo Kα can excite the Mo L shells (but not the K shell). Its L lines sit at
2.29–2.62 keV, on top of S Kα, Pb M and Tl M, so the Mo and S areas are strongly correlated:
read both with that in mind.

In [ ]:
CANDIDATES = ["Cl", "Ga", "Br", "Rb"]
t0 = time.time()
r_extra = fit_average_spectrum(y, make_groups(ELEMENTS + CANDIDATES), CAL0, DETECTOR0,
                               CAL0.channel_range(*ROI_EV, y.size), yerr=yerr, background_order=BKG_ORDER,
                               escape=True, fit_calibration=True, fit_tails=True, fit_gamma=False,
                               scatter=SCATTER, maxfev=40000)
print(f"reduced chi2 {result.chi2_red:.3f} -> {r_extra.chi2_red:.3f} with the candidates ({time.time()-t0:.0f} s)")
for g in r_extra.model.groups:
    if g.element in CANDIDATES:
        a, e = r_extra.areas[g.label], r_extra.area_errors[g.label]
        print(f"  {g.label:6s} area {a:8.0f} +/- {e:5.0f}   S/N {a/max(e, 1e-9):5.1f}")

Cl, Ga and Rb are absent; Br is at most marginal.

## 7. Check: concentrations from the Co internal standard

With Co at 500 ppm as internal standard, each element's concentration follows from its area
and its computed sensitivity (excitation × cross-section × self-absorption in the water ×
detection efficiency): `c_X = c_Co (A_X / S_X) / (A_Co / S_Co)`. The tutorial says every
element of the standard should come out near 500 ppm: this tests the whole chain of
calibration, line intensities, matrix and detector models.

In [ ]:
main = {}
for g in GROUPS:   # one family per element: the most precise
    snr = result.areas[g.label] / max(result.area_errors[g.label], 1e-9)
    if g.element not in main or snr > result.areas[main[g.element].label] / max(result.area_errors[main[g.element].label], 1e-9):
        main[g.element] = g
co = main["Co"]
k = result.areas[co.label] / co.ref_strength
CONC = {}
for el, g in main.items():
    c = CO_PPM * (result.areas[g.label] / g.ref_strength) / k
    rel = result.area_errors[g.label] / max(result.areas[g.label], 1e-9)
    CONC[el] = (g.label, c, c * rel)
print(f"{'element':8s}{'line':>7}{'ppm':>9}{'+/-':>7}")
for el, (lab, c, e) in CONC.items():
    note = {"Ar": "  (air)", "S": "  (overlaps Pb M, Tl M, Mo L)",
            "Mo": "  (L lines; overlaps S K, Pb M, Tl M; tube anode?)"}.get(el, "")
    print(f"{el:8s}{lab.split()[1]:>7}{c:9.0f}{e:7.0f}{note}")

std = [el for el in CONC if el not in ("Ar", "S", "K", "Ca", "Mo")]
fig, ax = plt.subplots(figsize=(11, 3.8))
z = [xraylib.SymbolToAtomicNumber(el) for el in std]
ax.errorbar(z, [CONC[el][1] for el in std], yerr=[CONC[el][2] for el in std], fmt="o", capsize=3)
for zi, el in zip(z, std):
    ax.annotate(el, (zi, CONC[el][1]), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=8)
ax.axhline(CO_PPM, color="0.5", lw=0.8); ax.axhspan(0.9 * CO_PPM, 1.1 * CO_PPM, color="0.9", zorder=0)
ax.set_xlabel("Z"); ax.set_ylabel("concentration (ppm)")
ax.set_title("Internal-standard concentrations (Co = 500 ppm); band: +/- 10 %", loc="left")
fig.tight_layout()

## 8. Save

In [ ]:
record = {
    "file": str(MCA_FILE), "excitation": {"anode": ANODE, "energies_ev": E_EXC.tolist(), "weights": W_EXC.tolist()},
    "roi_ev": list(ROI_EV), "background_order": BKG_ORDER, "chi2_red": result.chi2_red,
    "calibration": {"gain_ev_per_channel": CAL.gain, "offset_ev": CAL.offset,
                    "reference_peaks": found},
    "elements": ELEMENTS,
    "parameters": {k: {"value": v, "stderr": e} for k, (v, e) in P.items()},
    "areas": {g.label: {"area": result.areas[g.label], "error": result.area_errors[g.label]} for g in GROUPS},
    "concentrations_ppm_co_internal_standard": {el: {"line": lab, "ppm": c, "error": e} for el, (lab, c, e) in CONC.items()},
}
path = OUT_DIR / f"{MCA_FILE.stem}_fit.json"
path.write_text(json.dumps(record, indent=2, default=float))
print("saved", path)